# Environment Setup

In [1]:
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 225.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 316.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
%%capture
%pip install -U bitsandbytes
%pip install -U peft
%pip install -U accelerate
%pip install -U trl
%pip install -U datasets

In [3]:
%pip install datasets==2.16.0

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.21.0 requires datase

# Imports

In [4]:
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from random import randrange
from peft import LoraConfig, get_peft_model, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments,BitsAndBytesConfig
from trl import SFTTrainer

## Wandb

In [6]:
import wandb

# Log in to your W&B account
wandb.login()

# Initialize a new W&B run
wandb.init(
    project="sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa-train"
)

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abrahamkaikobadtushar (axiler) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Model & Tokenizer Loading

In [7]:
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login
import os

token = os.getenv('HF_TOKEN')
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face!


In [8]:
from dotenv import load_dotenv
load_dotenv()
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

# Dataset load and preprocess

In [9]:
import pandas as pd
cloud_train=pd.read_excel("../../data/splits/cloud/cloud_train.xlsx")

In [10]:
cloud_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4693 entries, 0 to 4692
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0.1     4693 non-null   int64 
 1   Unnamed: 0       4693 non-null   int64 
 2   title            4693 non-null   object
 3   body             4693 non-null   object
 4   tags             4693 non-null   object
 5   link             4693 non-null   object
 6   score            4693 non-null   int64 
 7   creation_date    4693 non-null   object
 8   answer_count     4693 non-null   int64 
 9   comments         4693 non-null   object
 10  answers          4693 non-null   object
 11  selected_answer  4693 non-null   object
 12  images           4693 non-null   object
dtypes: int64(4), object(9)
memory usage: 476.8+ KB


## ocr load

In [11]:
import json

with open("../../data/processed/ocr/cloud/ocr_extracted_cloud_train.jsonl", "r", encoding="utf-8") as file:
    cloud_train_ocr = [json.loads(line) for line in file]

In [12]:
from typing import List, Dict, Any

def ocr_extract(data) -> List[str]:
    results: List[str] = []

    for entry in data:
        lines=[]

        for img_i, img in enumerate(entry.get("images", [])):
            lines.append(f"img{img_i}:")

            for resp in img.get("response", []):
                if isinstance(resp, dict):
                    # --- non-table text ---
                    ntt = resp.get("non_table_text")
                    if ntt:
                        lines.append("Non-table text:")
                        lines.extend(ntt if isinstance(ntt, list) else [str(ntt)])

                    # --- tables ---
                    tables = resp.get("table")
                    if tables:
                        lines.append("Tables:")
                        for t_i, tbl in enumerate(tables if isinstance(tables, list) else [tables]):
                            lines.append(f"Table {t_i}:")
                            lines.append(tbl.get("content", "") if isinstance(tbl, dict) else str(tbl))
                    continue

                # Unknown or malformed type
                lines.append(f"[WARN] Unhandled response type: {type(resp).__name__}")
                lines.append(str(resp))

        formatted = "\n".join(lines).strip()
        if formatted:
            results.append(formatted)

    return results

In [13]:
# format ocr response for prompt
cloud_train_ocr_response_format = ocr_extract(cloud_train_ocr)   

In [14]:
len(cloud_train_ocr_response_format)

4693

In [15]:
cloud_train['images'][1234]

"['https://i.sstatic.net/bhQwK.png']"

In [16]:
print(cloud_train_ocr_response_format[1234])

img0:
Non-table text:
                → View as Backlog
Board
      Analytics
ve updates isn't available due to changes made to board layout. Please try to refresh the board.
                                 Doing
                                                            1/5
                                                                 Done
To Do
                            <
                                                                                             <
                            م
   New item
                                   26 Issue #1
                                            • Doing
                                  State
   27 Issue #2
 State
            • To Do


## processed

In [17]:
cloud_train=cloud_train[['title','body','selected_answer','images']]

In [18]:
cloud_train['context'] = "title: " + cloud_train['title'].astype(str) + "\nbody: " + cloud_train['body'].astype(str).str.strip()

In [19]:
cloud_train['selected_answer']=cloud_train['selected_answer'].str.strip()

In [20]:
cloud_train_processed=pd.DataFrame(
    {
        'question':cloud_train['context'],
        'answer':cloud_train['selected_answer']
    }
)

In [21]:
cloud_train_processed['ocr']=cloud_train_ocr_response_format

In [22]:
cloud_train_processed.head(2)

,question,answer,ocr
0,title: I am using Azure Devops to build and pu...,When using buildAndPush command in Docker task...,img0:\nNon-table text:\n ① Note\nThe arguments...
1,title: configure Amazon s3 bucket to run Lambd...,You can do this by providing the full Lambda F...,img0:\nNon-table text:\n Events\n Ev...


In [23]:
#cloud_train_processed=cloud_train_processed[:800]
#cloud_val_processed=cloud_val_processed[:100]

## prompt

In [24]:
zero_shot = '''
You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Answer:
{}
'''
print(zero_shot)
def formatting_func(row):
    return zero_shot.format(row['ocr'],row["question"],row["answer"])

cloud_train_processed["text"] = cloud_train_processed.apply(formatting_func, axis=1)


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Answer:
{}



In [25]:
print(cloud_train_processed["text"][1])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
       Events
     Event Notifications enable you to send alerts or trigger workflows. Notifications can be sent via Amazon
Simple Notification Service (SNS) or Amazon Simple Queue Service (SQS) or to a Lambda function
(depending on the bucket location).
                              e.g. MyEmailNotificationsForPut
                   Name
                                                                                                         Ⓡ
                              Select event(s)
                  Events
                                                                                                         Ⓡ
                              e.g. images/
                   Prefix
                              e.g. jpg
                   S

In [26]:
# Split 20% for validation, 80% for training
val_df = cloud_train_processed[["text"]].sample(frac=0.2, random_state=42)
train_df = cloud_train_processed[["text"]].drop(val_df.index)

In [27]:
from dotenv import load_dotenv
load_dotenv()
from datasets import Dataset, load_dataset

# Convert DataFrame to HF Dataset with just the 'text' column
trainset = Dataset.from_pandas(train_df.reset_index(drop=True))
valset = Dataset.from_pandas(val_df.reset_index(drop=True))

print(f"Training set size: {len(trainset)}")
print(f"Validation set size: {len(valset)}")

Training set size: 3754
Validation set size: 939


In [28]:
trainset,valset


(Dataset({
     features: ['text'],
     num_rows: 3754
 }),
 Dataset({
     features: ['text'],
     num_rows: 939
 }))

## Testing

In [25]:
# Test
sample_input = cloud_train_processed["text"].iloc[1]

In [ ]:
inputs = tokenizer(
    sample_input,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(model.device)

model.eval()
with torch.no_grad():
    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id
    )

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)

# Training Arguments

In [29]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM"
)

In [30]:
from dotenv import load_dotenv
load_dotenv()
# training args
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa",
    num_train_epochs=5,
    per_device_train_batch_size=3,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    gradient_checkpointing=True,
    seed=42,
    push_to_hub=True,
    load_best_model_at_end=True,
    hub_model_id="Bakugo123/sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa",
    hub_strategy="end",
)

# Trainer

In [31]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [32]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=trainset,
    eval_dataset=valset,
    peft_config=peft_config,
    #tokenizer=tokenizer,
    args=args,
    #max_seq_length=1024,
    #dataset_text_field="text"
)

Adding EOS to train dataset:   0%|          | 0/3754 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3754 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3754 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/939 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/939 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/939 [00:00<?, ? examples/s]

In [33]:
# Train
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
50,2.266000,2.290065
100,2.140000,2.055774
150,1.949200,1.907732
200,1.872900,1.858595
250,1.799900,1.823358
300,1.765900,1.808981
350,1.750000,1.797462
400,1.762600,1.791095
450,1.770600,1.786457
500,1.724400,1.783293


TrainOutput(global_step=785, training_loss=1.8294955988598478, metrics={'train_runtime': 63096.7566, 'train_samples_per_second': 0.297, 'train_steps_per_second': 0.012, 'total_flos': 8.236799247538668e+17, 'train_loss': 1.8294955988598478})

In [34]:
# Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

Evaluation Results: {'eval_loss': 1.7782957553863525, 'eval_runtime': 874.6418, 'eval_samples_per_second': 1.074, 'eval_steps_per_second': 0.135}


In [ ]:
# Save the model and tokenizer
#trainer.save_model("sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa")
#tokenizer.save_pretrained("sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa")

# Save Fine Tune Model

In [35]:
trainer.push_to_hub()

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...oud-zero-with-ocr-qa/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...th-ocr-qa/adapter_model.safetensors:  23%|##3       | 25.2MB /  109MB            

  ...-zero-with-ocr-qa/training_args.bin:   3%|2         |   166B / 6.22kB            

CommitInfo(commit_url='https://huggingface.co/Bakugo123/sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa/commit/2cba12fc1ab82d4c95416d38594cc8c83ab9d4a7', commit_message='End of training', commit_description='', oid='2cba12fc1ab82d4c95416d38594cc8c83ab9d4a7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Bakugo123/sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa', endpoint='https://huggingface.co', repo_type='model', repo_id='Bakugo123/sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa'), pr_revision=None, pr_num=None)

In [40]:
print(trainer.model)  

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [38]:
torch.cuda.empty_cache()

## Load fine tune model

In [ ]:
#model_id="Bakugo123/model_name" #fine tune model id

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import torch
from transformers import pipeline

# Check if a CUDA-compatible GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize the text-generation pipeline with the specified device
generator = pipeline("text-generation", model="Bakugo123/sft-llama3.1-8b-instruct-cloud-zero-with-ocr-qa", device=device)

# Test the generator
question = "You are cloud expert.what is gcp?"
output = generator(question, max_new_tokens=128, return_full_text=False)[0]
print(output["generated_text"])

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 | Google Cloud Platform (GCP) explained
Google Cloud Platform (GCP) is a suite of cloud computing services offered by Google. It provides a range of services including computing power, storage, networking, data analytics, machine learning, and more. GCP is designed to help businesses build, deploy, and manage applications and workloads in the cloud.
GCP offers a wide range of services, including:
Compute Services:
Cloud Functions: A serverless compute service that allows you to run small code snippets in response to events.
Cloud Run: A fully managed platform for building and deploying containerized applications.
App Engine: A managed platform
